# Module 9: Reporting & Dashboard

**QuantVerse** — Quantitative Portfolio Intelligence System

---

## Objectives

Consolidate all analysis into publication-quality outputs:

1. **Strategy Tearsheets** — one-page quant tearsheets per strategy
2. **Multi-Strategy Dashboard** — 7-panel summary comparing all strategies
3. **Executive Summary Report** — key findings, rankings, recommendations
4. **Export** — PNG tearsheets, dashboard, text report

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings, logging, sys, os, json

sys.path.insert(0, os.path.abspath('..'))
warnings.filterwarnings('ignore')
logging.basicConfig(level=logging.INFO)

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 12
sns.set_palette('husl')
print('Setup complete.')

In [ ]:
# Load data
data_dir = '../data/processed'
daily_returns = pd.read_parquet(f'{data_dir}/returns_daily.parquet')
clean_prices = pd.read_parquet(f'{data_dir}/prices_clean.parquet')

with open(f'{data_dir}/asset_class_map.json', 'r') as f:
    class_map = json.load(f)

signal_tickers = [t for t, c in class_map.items() if c == 'signals']
investable = [t for t in daily_returns.columns if t not in signal_tickers]
returns = daily_returns[investable].dropna()

# Weights
weights_path = f'{data_dir}/portfolio_weights.parquet'
if os.path.exists(weights_path):
    all_weights = pd.read_parquet(weights_path)
else:
    all_weights = pd.DataFrame({'Equal Weight': pd.Series(1/len(investable), index=investable)})

w_ew = pd.Series(1/len(investable), index=investable)
bench_ret = pd.Series(returns.values @ w_ew.values, index=returns.index, name='Equal Weight')

print(f'Assets: {len(investable)}, Strategies: {list(all_weights.columns)}')

## 1. Strategy Tearsheets

In [ ]:
from project.reporting import TearsheetGenerator

# Output directory
output_dir = '../reports'
os.makedirs(output_dir, exist_ok=True)

# Generate tearsheet for primary strategy
primary = 'Max Sharpe' if 'Max Sharpe' in all_weights.columns else all_weights.columns[0]
w_primary = all_weights[primary]
port_ret = pd.Series(returns.values @ w_primary.reindex(investable).fillna(0).values,
                      index=returns.index)

ts = TearsheetGenerator(port_ret, benchmark=bench_ret, name=primary)
fig = ts.generate(save_path=f'{output_dir}/tearsheet_{primary.lower().replace(" ", "_")}.png')
plt.show()

In [ ]:
# Generate tearsheets for all strategies
for strat in all_weights.columns:
    w = all_weights[strat].reindex(investable).fillna(0)
    r = pd.Series(returns.values @ w.values, index=returns.index)
    ts = TearsheetGenerator(r, benchmark=bench_ret, name=strat)
    fig = ts.generate(save_path=f'{output_dir}/tearsheet_{strat.lower().replace(" ", "_")}.png')
    plt.close(fig)
    print(f'  ✓ Tearsheet: {strat}')

print(f'\n{len(all_weights.columns)} tearsheets saved to {output_dir}/')

## 2. Multi-Strategy Dashboard

In [ ]:
from project.reporting import DashboardDataBuilder

dash = DashboardDataBuilder(returns, all_weights, class_map)
fig = dash.plot_dashboard(save_path=f'{output_dir}/dashboard.png')
plt.show()

In [ ]:
# Risk-Return data
rr = dash.risk_return_scatter()
print('Risk-Return Summary:')
print('=' * 70)
print((rr * 100).round(2).to_string())

In [ ]:
# Asset class allocation
alloc = dash.allocation_summary()
print('\nAsset Class Allocation (%):')
print('=' * 90)
print(alloc.round(1).to_string())

In [ ]:
# Strategy correlation
corr = dash.strategy_correlation()
print('\nStrategy Return Correlations:')
print('=' * 70)
print(corr.round(3).to_string())

## 3. Executive Summary Report

In [ ]:
from project.reporting import ReportGenerator

rg = ReportGenerator(data_dir=data_dir)
report_text = rg.generate_text_report()

# Save to file
with open(f'{output_dir}/executive_report.txt', 'w') as f:
    f.write(report_text)

print(report_text)

In [ ]:
# Strategy ranking
ranking = rg.strategy_ranking()
if len(ranking) > 0:
    print('\nStrategy Ranking (lower = better):')
    print(ranking.round(1).to_string())

## 4. Additional Tearsheet — Equal Weight (Benchmark)

In [ ]:
ts_ew = TearsheetGenerator(bench_ret, name='Equal Weight (1/N) Benchmark')
fig = ts_ew.generate(save_path=f'{output_dir}/tearsheet_equal_weight_benchmark.png')
plt.show()

## 5. Final Output Summary

In [ ]:
import glob

outputs = glob.glob(f'{output_dir}/*')
print('Generated Reports:')
print('=' * 60)
for f in sorted(outputs):
    size = os.path.getsize(f) / 1024
    print(f'  {os.path.basename(f):45s} {size:>8.1f} KB')

---

## QuantVerse — All 9 Modules Complete

### Module Summary

| Module | Description | Key Output |
|--------|-------------|------------|
| **1** | Data Pipeline | 41-asset, multi-class daily data |
| **2** | Exploratory Analysis | Distribution fitting, stylized facts, GARCH |
| **3** | Covariance Estimation | 7 estimators compared, Ledoit-Wolf default |
| **4** | Portfolio Optimization | 8 strategies: MV, BL, HRP, RP, CVaR |
| **5** | Risk Analysis | VaR/CVaR (4 methods), drawdown, tail risk, factor decomposition |
| **6** | Monte Carlo & Stress | 10K simulations, 13 historical + 6 hypothetical scenarios |
| **7** | Backtesting | Walk-forward, Brinson-Fachler attribution, rebalancing |
| **8** | Regime Detection | HMM, K-Means, Vol regimes, adaptive allocation |
| **9** | Reporting | Tearsheets, dashboard, executive report |

### Key Deliverables

- **Per-strategy tearsheets** (PNG) — institutional-quality one-pagers
- **Multi-strategy dashboard** (PNG) — 7-panel comparative view
- **Executive report** (TXT) — key findings, rankings, recommendations
- **All intermediate data** (Parquet) — for further analysis

---